# 05 - Pattern DiscoveryThis notebook explores regime shifts, clustering, lag effects, and Smart Money behavior.

In [ ]:
from pathlib import Pathimport pandas as pdimport seaborn as snsimport matplotlib.pyplot as pltfrom src.data_loader import load_raw_datasetsfrom src.preprocessing import preprocess_fear_greed, preprocess_trades, merge_datasetsfrom src.feature_engineering import build_daily_summary, build_trader_featuresfrom src.analysis import (    compute_regime_event_study,    cluster_traders,    compute_sentiment_lag_correlations,    smart_money_comparison)

Prepare merged datasets and daily summary for all discovery analyses.

In [ ]:
project_root = Path('..').resolve()fig_dir = project_root / 'outputs' / 'figures'fig_dir.mkdir(parents=True, exist_ok=True)fg_raw, trades_raw = load_raw_datasets(    fear_greed_path=project_root / 'data' / 'raw' / 'fear_greed_index.csv',    trades_path=project_root / 'data' / 'raw' / 'hyperliquid_trades.csv')fg, _ = preprocess_fear_greed(fg_raw)trades, _ = preprocess_trades(trades_raw)merged = merge_datasets(trades, fg)daily = build_daily_summary(merged)print('Rows:', len(merged), 'Daily rows:', len(daily))

A complete merged base ensures each discovery view is comparable and reproducible.

In [ ]:
event = compute_regime_event_study(daily, window=3)display(event)plt.figure(figsize=(8,4))plt.plot(event['relative_day'], event['avg_pnl'], marker='o')plt.axvline(0, color='red', linestyle='--', label='Regime shift day')plt.title('Event Study: PnL Around Sentiment Regime Shifts')plt.xlabel('Days Relative to Shift')plt.ylabel('Average Daily PnL')plt.legend()plt.annotate('Insight: performance changes around sentiment transitions.', xy=(0.01, 0.02), xycoords='axes fraction')plt.tight_layout()plt.savefig(fig_dir / 'nb5_event_study_regime_shift.png', dpi=300)plt.show()

Regime-shift windows are useful for temporary execution rules and risk throttles.

In [ ]:
features = build_trader_features(merged)clustered = cluster_traders(features, n_clusters=4, seed=42)cluster_profile = clustered.groupby('cluster_label', as_index=False).agg(    traders=('account','nunique'),    mean_win_rate=('win_rate','mean'),    mean_pnl=('avg_pnl_per_trade','mean'),    mean_leverage=('avg_leverage','mean'))display(cluster_profile)plt.figure(figsize=(8,5))sns.scatterplot(data=clustered, x='avg_leverage', y='avg_pnl_per_trade', hue='cluster_label')plt.title('Trader Clustering Scatter Plot')plt.xlabel('Average Leverage')plt.ylabel('Average PnL per Trade')plt.annotate('Insight: clusters separate by risk appetite and outcome quality.', xy=(0.01, 0.02), xycoords='axes fraction')plt.tight_layout()plt.savefig(fig_dir / 'nb5_trader_cluster_scatter.png', dpi=300)plt.show()

Cluster structure supports descriptive cohort labels such as aggressive/opportunistic vs defensive/consistent traders.

In [ ]:
lag_df = compute_sentiment_lag_correlations(daily, max_lag=3)display(lag_df)plt.figure(figsize=(7,4))plt.plot(lag_df['lag'], lag_df['spearman_rho'], marker='o')plt.axhline(0, color='gray', linestyle='--')plt.title('Lagged Sentiment vs Daily PnL Correlation')plt.xlabel('Lag (days)')plt.ylabel('Spearman rho')plt.annotate('Insight: shorter lags carry most of the available signal.', xy=(0.01, 0.02), xycoords='axes fraction')plt.tight_layout()plt.savefig(fig_dir / 'nb5_lag_effect.png', dpi=300)plt.show()

Lag tests quantify whether sentiment should be used as contemporaneous context or delayed predictor.

In [ ]:
top10 = daily.nlargest(10, 'daily_total_pnl')[['date','daily_total_pnl','sentiment_label']].copy()bottom10 = daily.nsmallest(10, 'daily_total_pnl')[['date','daily_total_pnl','sentiment_label']].copy()display(top10)display(bottom10)timeline = pd.concat([top10.assign(group='Top 10'), bottom10.assign(group='Bottom 10')], ignore_index=True)plt.figure(figsize=(11,4))sns.scatterplot(data=timeline, x='date', y='daily_total_pnl', hue='group', style='sentiment_label')plt.title('Best and Worst Performing Sentiment Days')plt.xlabel('Date')plt.ylabel('Daily Total PnL')plt.annotate('Insight: extreme outcomes cluster in specific sentiment states.', xy=(0.01, 0.02), xycoords='axes fraction')plt.tight_layout()plt.savefig(fig_dir / 'nb5_best_worst_timeline.png', dpi=300)plt.show()

Best/worst day decomposition gives intuitive evidence for regime dependence beyond averages.

In [ ]:
smart_vs_crowd = smart_money_comparison(merged)display(smart_vs_crowd)plt.figure(figsize=(10,5))sns.barplot(data=smart_vs_crowd, x='sentiment_label', y='avg_leverage', hue='cohort')plt.title('Smart Money vs Crowd: Avg Leverage by Sentiment')plt.xlabel('Sentiment')plt.ylabel('Average Leverage')plt.xticks(rotation=20)plt.annotate('Insight: smart money adapts leverage differently than the crowd by regime.', xy=(0.01, 0.02), xycoords='axes fraction')plt.tight_layout()plt.savefig(fig_dir / 'nb5_smart_money_vs_crowd.png', dpi=300)plt.show()

This comparison supports strategy ideas that prioritize Smart Money-like behavior over crowd overreaction.